# Block 4 — Time Series Data with DARTS

**Goals for this block:**
- Introduction to the Darts Library for time series forecasting
- Build and train SARIMA model using DARTS
- Build and train PROPHET model using Darts


## 0. Setup & Environment


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

## 1. Data Loading & Preparation

We'll work with the **air passengers** dataset from Block 1. 

In [ ]:
# GUIDE: load and preprocess data

df = pd.read_csv("air_passengers.csv")
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.index.freq = 'MS'  # monthly start frequency
df.set_index('timestamp', inplace=True)

## 2. Training SARIMA Models with Darts

Now we'll explore the Darts library, which offers specialized time series functionality and simplifies the implementation of advanced forecasting models.

In [ ]:
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.utils.missing_values import fill_missing_values

In [ ]:
# Convert the DataFrame to a Darts TimeSeries object
series = TimeSeries.from_dataframe(df)

#### 2.2. Preprocess Data 

In [ ]:
# 1. Handle missing values
filled_series = fill_missing_values(series, method="linear")

# 2. Resample data
resampled_series = filled_series.resample("MS", method="sum")

# 3. Time-aware split

# Calculate split points
train_series, test_series = resampled_series.split_before(0.7)

# 4. Scaling (fit only on training data)
scaler = Scaler(StandardScaler())
train_scaled = scaler.fit_transform(train_series)
test_scaled = scaler.transform(test_series)

In [ ]:
orignal_data = resampled_series
preprocessed_train = train_scaled
preprocessed_test = test_scaled

#### 2.3. build Naive Forecaster as Baseline

In [ ]:
naive_forecaster_pred_scaled = preprocessed_test.shift(1)

#### 2.4. Train the SARIMA Model

Darts provides a convenient wrapper for the `statsmodels` ARIMA and `StatsForecastModel` AutoARIMA implementation.

We will:
1.  Instantiate the `AutoARIMA` model, specifying the seasonal period (`season_length=12`).
2.  Fit the model on the scaled training data.
3.  Generate a forecast and inverse-transform it.

In [ ]:
from darts.models import ARIMA
from darts.models import AutoARIMA

# # 1. Define and fit the SARIMA model
# # For monthly data with yearly seasonality, m=12
# # We use common (p,d,q)(P,D,Q,m) orders for this dataset
# model_sarima = ARIMA(p=1, d=1, q=1, seasonal_order=(1, 1, 0, 12))
# model_sarima.fit(preprocessed_train)

# 1. Define and fit the AutoARIMA model
# We only need to specify the seasonality period (mseason_length=12)
model_auto_arima = AutoARIMA(season_length=12, stepwise=True)
model_auto_arima.fit(preprocessed_train)

In [ ]:
# 2. Generate forecasts
n_forecast = len(preprocessed_test)
pred_auto_arima_scaled = model_auto_arima.predict(n=n_forecast)

# 3. Inverse-transform the forecast to the original scale
pred_auto_arima = scaler.inverse_transform(pred_auto_arima_scaled)
naive_forecaster_pred = scaler.inverse_transform(naive_forecaster_pred_scaled)

# 5. Plot the results
plt.figure(figsize=(12, 6))
orignal_data.plot(label="Actual")
pred_auto_arima.plot(label="AutoARIMA Forecast")
naive_forecaster_pred.plot(label="Naive Forecast")
plt.xlabel("Timestamp")
plt.ylabel("Number of Passengers")
plt.legend()

## 3. Model Evaluation

Now that we have forecasts from both a `ARIMA` model and a simple `Naive` baseline, we need to quantitatively evaluate their performance. This allows us to determine if the more complex model provides a meaningful improvement.

We will use the following standard error metrics:
-   **MAE (Mean Absolute Error)**: Measures the average absolute difference between forecasts and actuals. It's easy to understand and less sensitive to large outliers than RMSE.
-   **RMSE (Root Mean Squared Error)**: Measures the square root of the average squared errors. It penalizes large errors more heavily.
-   **MAPE (Mean Absolute Percentage Error)**: Measures the average percentage error. It's intuitive but can be problematic if actual values are close to zero.
-   **sMAPE (Symmetric Mean Absolute Percentage Error)**: An alternative to MAPE that is bounded between 0% and 200%, making it less biased when actual values are small.

In [ ]:
from darts.metrics import mae, rmse, mape, smape

# Calculate metrics for the ARIMA model
mae_auto_arima = mae(test_series, pred_auto_arima)
rmse_auto_arima = rmse(test_series, pred_auto_arima)
mape_auto_arima = mape(test_series, pred_auto_arima)
smape_auto_arima = smape(test_series, pred_auto_arima)

# Calculate metrics for the Naive model
mae_naive = mae(test_series, naive_forecaster_pred)
rmse_naive = rmse(test_series, naive_forecaster_pred)
mape_naive = mape(test_series, naive_forecaster_pred)
smape_naive = smape(test_series, naive_forecaster_pred)

In [ ]:
results = {
    'AutoARIMA': {
        'MAE': mae_auto_arima,
        'RMSE': rmse_auto_arima,
        'MAPE (%)': mape_auto_arima,
        'sMAPE (%)': smape_auto_arima
    },
    'Naive Forecast': {
        'MAE': mae_naive,
        'RMSE': rmse_naive,
        'MAPE (%)': mape_naive,
        'sMAPE (%)': smape_naive
    }
}
results_df = pd.DataFrame(results).round(2)
results_df

<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: train and evaluate using darts</h2>

Apply what you've learned to the electricity consumption dataset:
1. **Load the data**: Load `electricity_consumptions.csv`.
2. **Remove unnecessary columns**: You should see an indices column "Unnamed: 0" which can be dropped
3. **Split Train/Test and rescale**: 70% Train, 30% Test and use a scaling method
2. **Build and Train**:
   - Are the data hourly/daily/monthly? What is the seasonal period of the data?
   - Create and fit a `Auto_ARIMA` model using DARTS.
3. **Forecast and Visualize**:
   - Generate predictions for the test set.
   - Plot the actual values, your SARIMA forecast, and a simple Naive forecast on the same graph.
4. **Evaluate**:
   - Compare the results. Did the SARIMA model improve upon the baseline?

This exercise will learn building and evaluating using DARTS.
</div>

In [ ]:
# load data



In [ ]:
# drop unnecessary columns using df.drop(columns=...


In [ ]:
# Plot the data. Try to zoom in to analyze the data 


In [ ]:
# convert dataframe to TimeSeries


In [ ]:
# Train/Test Split


In [ ]:
# Scaling the train and test sets


In [ ]:
# Naive Forecast


In [ ]:
# Define and fit the Auto model


In [ ]:
# 2. Generate forecasts


In [ ]:
# calculate metrics for the AutoARIMA model


# calculate metrics for the Naive model


## 4. Training a Prophet Model

Next, we'll train a Prophet model. Prophet is particularly effective for time series that have strong seasonal effects and several seasons of historical data. It is robust to missing data and shifts in the trend, and typically handles outliers well.

`seasonality_mode = Multiplicative` means we have a Multiplicative model (trend × (1+ seasonality)) which has a Seasonality `scales` with the trend.

`changepoint_prior_scale = 0.5` allows stronger and more frequent changes. This hyperparameter defines how easily Prophet adjusts its trend when it detects a structural change. Moderately high changepoint_prior_scale lets Prophet capture the accelerating upward trend without overfitting.

In [ ]:
from darts.models import Prophet

# 1. Define and fit the Prophet model
# Prophet handles scaling and seasonality internally, but we use our preprocessed data for consistency

model_prophet = Prophet(
     seasonality_mode='multiplicative',   
     changepoint_prior_scale= 0.2,
     yearly_seasonality=True,
)

# model_prophet = Prophet(
# )
model_prophet.fit(preprocessed_train)

In [ ]:
# 2. Generate forecasts
n_forecast = len(preprocessed_test)
pred_prophet_scaled = model_prophet.predict(n=n_forecast)

In [ ]:
# 3. Inverse-transform the forecast to the original scale
pred_prophet = scaler.inverse_transform(pred_prophet_scaled)

In [ ]:
plt.figure(figsize=(12, 6))
orignal_data.plot(label="Actual")
pred_auto_arima.plot(label="AutoARIMA Forecast")
naive_forecaster_pred.plot(label="Naive Forecast")
pred_prophet.plot(label="Prophet Forecast")

plt.title("Model Forecast Comparison")
plt.xlabel("Timestamp")
plt.ylabel("Number of Passengers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Calculate metrics for the Prophet model
mae_prophet = mae(test_series, pred_prophet)
rmse_prophet = rmse(test_series, pred_prophet)
mape_prophet = mape(test_series, pred_prophet)
smape_prophet = smape(test_series, pred_prophet)

In [ ]:
results['Prophet'] = {
    'MAE': mae_prophet,
    'RMSE': rmse_prophet,
    'MAPE (%)': mape_prophet,
    'sMAPE (%)': smape_prophet
}

results_df = pd.DataFrame(results).round(2)
results_df

</VSCode.Cell>
<VSCode.Cell id="#VSC-c848b7c8" language="markdown">
<div style="background-color: #ffedcc; border-left: 6px solid #ff7518; padding: 10px; margin-bottom: 10px;">
<h2>📝 Exercise: Train and Evaluate a Prophet Model</h2>

Now, apply the Prophet model to the `electricity_consumption.csv` dataset that you worked with in the previous exercise.

1.  **Build and Train**:
    *   Instantiate and fit a `Prophet` model using Darts on the training data for electricity consumption.
2.  **Forecast and Visualize**:
    *   Generate predictions for the test set.
    *   Update your previous plot to include the Prophet forecast alongside the actuals, the AutoARIMA forecast, and the Naive forecast.
3.  **Evaluate**:
    *   Calculate the evaluation metrics (MAE, RMSE, MAPE, sMAPE) for the Prophet model.

This exercise will complete your comparison of baseline, statistical, and automated forecasting models on a new dataset.
</div>
</VSCode.Cell>
<VSCode.Cell id="#VSC-9dbb9b49" language="markdown">
</VSCode.Cell>

In [ ]:
# build and fit the Prophet model for the electricity consumption data
# Do you have here a "daily_seasonality"? "weekly_seasonality"? "yearly_seasonality"?

model_prophet_ex = ...

In [ ]:
# Generate forecasts



In [ ]:
# visualize all model forecasts


In [ ]:
# Evaluate Prophet model for electricity consumption data


## ✅ Summary 

### What You've Accomplished:

-   **Leveraged the Darts Library**: You were introduced to Darts, a powerful and user-friendly library for time series forecasting, and learned how to use its core `TimeSeries` object.
-   **Streamlined Model Training**: You experienced the unified API of Darts by training both an `AutoARIMA` and a `Prophet` model with a consistent workflow.
-   **Built and Evaluated Multiple Models**: You successfully built, trained, and generated forecasts for `AutoARIMA`, `Prophet`, and a `Naive` baseline on two different datasets.
-   **Performed Quantitative Comparison**: You used Darts' integrated metrics (MAE, RMSE, MAPE, sMAPE) to quantitatively evaluate and compare the performance of different forecasting models, identifying the best approach for each dataset.

You are now equipped to use Darts to efficiently tackle a wide range of time series forecasting problems.


## High Frequency Data

In [ ]:
from darts.datasets import ElectricityConsumptionZurichDataset

In [ ]:
data_high_freq = ElectricityConsumptionZurichDataset().load() ["Value_NE5"]

data_high_freq.to_dataframe().head(10)

In [ ]:
plt.figure(figsize=(12, 6))
data_high_freq.plot(title="Electricity Consumption - High Frequency Data (Zurich)")

In [ ]:
data_high_freq[:96*2].plot()

In [ ]:
# Preprocess the high-frequency data

# 1. Time-aware split

# Calculate split points
train_series, test_series = data_high_freq.split_before(0.7)

# 2. Scaling (fit only on training data)
scaler = Scaler(StandardScaler())
train_scaled = scaler.fit_transform(train_series)
test_scaled = scaler.transform(test_series)

In [ ]:
# # Train AutoARIMA model on high-frequency data
# model_auto_arima = AutoARIMA(season_length=96, stepwise=True)
# model_auto_arima.fit(train_scaled)

In [ ]:
model_prophet = Prophet()
model_prophet.fit(train_scaled)

In [ ]:
n_forecast = len(test_scaled)
pred_prophet_scaled = model_prophet.predict(n=n_forecast)

In [ ]:
pred_prophet = scaler.inverse_transform(pred_prophet_scaled)
plt.figure(figsize=(12, 6))
test_series.plot(label="Actual")
pred_prophet.plot(label="Prophet Forecast")
plt.title("Model Forecast")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# calculate metrics for the Prophet model
mae_prophet = mae(test_series, pred_prophet)
rmse_prophet = rmse(test_series, pred_prophet)
mape_prophet = mape(test_series, pred_prophet)
smape_prophet = smape(test_series, pred_prophet)

results = {
    'Prophet': {
        'MAE': mae_prophet,
        'RMSE': rmse_prophet,
        'MAPE (%)': mape_prophet,
        'sMAPE (%)': smape_prophet
    }
}
results_df = pd.DataFrame(results).round(2)
results_df